# 02 — Perfiles de voz con XTTS-v2

Nuestra solución extrae de cada grabación las representaciones necesarias para generar respuestas con una voz propia o autorizada. No entrenamos XTTS desde cero: cada perfil se guarda una vez y se reutiliza en la integración. No hace falta crear un archivo de transcripción `.txt`.


In [ ]:
%pip -q install -r requirements-lab4-modular.txt

## 1. Preparación del entorno y comprobación de CUDA

Este notebook prepara perfiles de voz en GPU, aislado de la integración para no competir con Qwen por VRAM. CUDA se comprueba antes de cargar XTTS; el notebook no cambia silenciosamente a CPU.


In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activa CUDA y usa el entorno .venv-dl-cuda para preparar perfiles de voz.")
torch.cuda.init()
torch.arange(1, device="cuda")
torch.cuda.synchronize()
DEVICE = "cuda:0"
print("Dispositivo:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM libre: {torch.cuda.mem_get_info(0)[0] / 1024**3:.1f} GB")


## 2. Cargar el modelo XTTS-v2

Se carga el modelo multilingüe XTTS-v2. La primera ejecución puede descargarlo y tardar varios minutos. Luego se reutiliza la copia local. El modelo convierte los textos del asistente en audio.

La celda también configura la carga del checkpoint requerida por PyTorch.

In [ ]:
import os
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.api import TTS

# XTTS-v2 oficial serializa estas clases de configuración en su checkpoint.
torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig])
os.environ.setdefault("COQUI_TOS_AGREED", "1")

# Compatibilidad acotada a la carga del checkpoint XTTS oficial; luego se restaura.
_anterior_weights_only = os.environ.get("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
try:
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
finally:
    if _anterior_weights_only is None:
        os.environ.pop("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", None)
    else:
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = _anterior_weights_only

model = tts.synthesizer.tts_model
print("XTTS-v2 cargado")


## 3. Seleccionar la identidad y la grabación de referencia

Detectamos las grabaciones existentes en `multimedia/multimediaNNN/referencia.wav` (identidad en `meta.json`) y en `data/voice_samples/<voz>.wav`. El menú permite escoger la voz para escuchar una síntesis de prueba; el notebook genera un `.pt` por cada perfil detectado.

Usa una grabación propia o autorizada, con una sola persona, voz natural y poco ruido. No es necesario crear un `.txt`. Si `meta.json` ya contiene la frase de referencia, se conserva como dato opcional para evitar repetirla literalmente en la síntesis.


In [ ]:
from pathlib import Path
import json
import sys
import ipywidgets as widgets
from IPython.display import display

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "lab4_asistente").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio (src/lab4_asistente).")
sys.path.insert(0, str(PROJECT_DIR / "src"))
from lab4_asistente import config as C

VOICE_PROFILES_DIR = C.RUTA_VOICE_PROFILES
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
VOICES = C.voces_disponibles()
if not VOICES:
    raise FileNotFoundError("Agrega una grabación propia/autorizada en multimedia/multimediaNNN/referencia.wav o data/voice_samples/<voz>.wav.")
voice_selector = widgets.Dropdown(options=sorted(VOICES), description="Perfil:", style={"description_width": "initial"})
display(voice_selector)
print("Perfiles de voz detectados:", ", ".join(sorted(VOICES)))
print("La selección define el nombre del archivo .pt generado.")


## 4. Extraer las representaciones de voz

XTTS calcula `gpt_cond_latent` y `speaker_embedding`. El bucle procesa todos los WAV y guarda un perfil por voz. XTTS-v2 sigue siendo necesario para sintetizar el audio.

In [ ]:
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capacidad GPU:", torch.cuda.get_device_capability(0))
    print("Arquitecturas de PyTorch:", torch.cuda.get_arch_list())

if DEVICE.startswith("cuda"):
    print("Prueba básica de CUDA: OK")
else:
    raise RuntimeError("La preparación de perfiles requiere CUDA.")

In [ ]:
generated_profiles = []
failed_profiles = []

for voice_id, voice_info in VOICES.items():
    voice_sample_path = voice_info["referencia"]
    profile_path = VOICE_PROFILES_DIR / f"{voice_id}.pt"
    try:
        with torch.inference_mode():
            gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
                audio_path=[str(voice_sample_path)],
                gpt_cond_len=6,
                max_ref_length=30,
                sound_norm_refs=False,
            )
        torch.save({
            "voice_id": voice_id,
            "reference_text": voice_info.get("texto_referencia", ""),
            "gpt_cond_latent": gpt_cond_latent.detach().cpu(),
            "speaker_embedding": speaker_embedding.detach().cpu(),
            "model": "XTTS-v2",
            "language": "es",
        }, profile_path)
        generated_profiles.append(profile_path)
        print(f"✓ {voice_id}: {profile_path.name}")
    except Exception as exc:
        failed_profiles.append((voice_id, str(exc)))
        print(f"✗ {voice_id}: {exc}")

print(f"\nPerfiles generados: {len(generated_profiles)}/{len(VOICES)}")
if failed_profiles:
    print("Revisar:", failed_profiles)


## 5. Persistir el artefacto generado

Cada perfil se guarda como `data/voice_profiles/<VOICE_ID>.pt`. Incluye la identidad, las representaciones de voz, el modelo y el idioma. El resultado muestra qué perfiles se generaron.

Los audios y perfiles contienen información de voz. Manténlos en un espacio privado.

In [ ]:
print("Perfiles disponibles para el asistente:")
for profile_path in sorted(VOICE_PROFILES_DIR.glob("*.pt")):
    print(" -", profile_path.stem, "→", profile_path.name)

## 6. Validar la síntesis

La prueba carga un perfil y convierte un texto nuevo en audio. La calidad depende del ruido, la duración y la expresividad del audio de referencia.

In [ ]:
VOICE_ID = voice_selector.value
profile_path = VOICE_PROFILES_DIR / f"{VOICE_ID}.pt"
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = "Hola, quiero realizar compras en línea y necesito ayuda para encontrar los mejores productos."
with torch.inference_mode():
    result = model.inference(
        text=text, language="es",
        gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
        speaker_embedding=profile["speaker_embedding"].to(DEVICE),
        enable_text_splitting=True,
    )
from IPython.display import Audio, display
display(Audio(result["wav"], rate=24000))
